# Week 7: random-window training and multi-window inference

The binary research showed that a record can be longer than 10 seconds. Here, the 16-base-code multilabel ResNet is trained with a **uniform random 10-second window** from each training ECG on each encounter. Its checkpoint is selected by the same first-window validation BCE used by the unchanged Week 7 baseline. We compare both models on identical first 10-second validation windows. A secondary, exploratory comparison scores both checkpoints on three deterministic windows (start, middle, end), aggregated by mean or maximum probability. No test labels or waveforms are accessed.

The 19 exact ICD strings and their preserved mapping to 16 base targets remain the label source. A zero target means the code is not among deposited selected annotations, not verified clinical absence. The split is patient-level; repeated ECGs from a patient are correlated. Multi-window results are validation observations only, not a new selected model or test result.

In [1]:
from pathlib import Path
import csv, random, re, json, os, sys, platform, importlib.util
from collections import Counter
import numpy as np
import pandas as pd
import torch, wfdb
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'Data/AttributesDictionary.csv').is_file())
OUT=ROOT/'new work/Week7/experimental/window_augmentation'
BASE=ROOT/'new work/Week7/experimental/multilabel_base16'
assert OUT.is_dir() and BASE.is_dir()
assert Path(sys.executable).resolve()==(ROOT/'venv/bin/python').resolve(), 'Use existing project venv kernel'
assert torch.cuda.is_available(), 'CUDA unavailable in actual notebook kernel; stop before training'
DEVICE=torch.device('cuda:0')
assert (torch.ones(1,device=DEVICE)*2).item()==2
print('Python',sys.executable,platform.python_version(),'Torch',torch.__version__,'CUDA',torch.cuda.get_device_name(0))
SEED=42;FS=500;N_SAMPLES=5000;BATCH_SIZE=32;MAX_EPOCHS=8;PATIENCE=3;LR=3e-4;WEIGHT_DECAY=1e-4
LEADS=['I','II','III','AVR','AVL','AVF','V1','V2','V3','V4','V5','V6'];LI={x:i for i,x in enumerate(LEADS)}
MAP=json.loads((BASE/'target_mapping.json').read_text()); exact_codes=MAP['exact_codes'];base_codes=MAP['base_codes_in_model_order'];base_of=MAP['exact_to_base']
assert len(exact_codes)==19 and len(base_codes)==16 and len(set(base_codes))==16
base_index={x:i for i,x in enumerate(base_codes)};exact_set=set(exact_codes)
torch.set_num_threads(min(4,os.cpu_count() or 1))

Python /home/souhit/Documents/Research/Pediatric_ECG/venv/bin/python 3.13.5 Torch 2.6.0+cu124 CUDA NVIDIA GeForce RTX 3060 Laptop GPU


In [2]:
META=ROOT/'Data/AttributesDictionary.csv'
with META.open(newline='') as f:
 reader=csv.reader(f);h=next(reader);pi=h.index('Patient_ID');patient_ids=sorted({r[pi] for r in reader})
random.Random(SEED).shuffle(patient_ids)
a=int(.70*len(patient_ids));b=int(.15*len(patient_ids))
train_ids=set(patient_ids[:a]);val_ids=set(patient_ids[a:a+b]);test_ids=set(patient_ids[a+b:])
assert (len(train_ids),len(val_ids),len(test_ids))==(8150,1746,1747)
def tokens(x):return [v.strip().strip("'\"").strip() for v in x.split(';') if v.strip()]
train_rows=[];val_rows=[]
with META.open(newline='') as f:
 for r in csv.DictReader(f):
  pid=r['Patient_ID']
  if pid in test_ids:continue # no held-out ICD/age/lead/signal access
  assert pid in train_ids or pid in val_ids
  selected=set(tokens(r['ICD-10 code']))&exact_set
  y=np.zeros(16,dtype=np.float32)
  for x in selected:y[base_index[base_of[x]]]=1
  item={'patient_id':pid,'ecg_id':r['ECG_ID'],'filename':r['Filename'],'sampling_point':int(r['Sampling_point']),
        'age_years':int(r['Age'][:-1])/365.25,'lead_count':int(r['Lead']),'target':y}
  (train_rows if pid in train_ids else val_rows).append(item)
assert (len(train_rows),len(val_rows))==(9943,2169)
Y_train=np.stack([r['target'] for r in train_rows]);Y_val=np.stack([r['target'] for r in val_rows])
assert Y_train.shape==(9943,16) and Y_val.shape==(2169,16)
prior=pd.read_csv(BASE/'validation_predictions.csv',dtype={'patient_id':str,'ecg_id':str})
assert prior.patient_id.tolist()==[r['patient_id'] for r in val_rows]
assert prior.ecg_id.tolist()==[r['ecg_id'] for r in val_rows]
assert np.array_equal(prior[[f'target::{x}' for x in base_codes]].to_numpy(dtype=np.float32),Y_val)
counts=Y_train.sum(0);pos_weight=np.minimum((len(Y_train)-counts)/counts,20).astype(np.float32)
assert (counts>0).all()
print('Exact baseline validation rows and target matrix reproduced; positive weights from training only:')
display(pd.DataFrame({'code':base_codes,'train_positive_records':counts.astype(int),'pos_weight':pos_weight}).round(3))

Exact baseline validation rows and target matrix reproduced; positive weights from training only:


,code,train_positive_records,pos_weight
0,I40.0,296,20.000
1,I40.9,39,20.000
2,I51.4,106,20.000
3,I42.0,44,20.000
4,I42.2,18,20.000
5,I42.9,17,20.000
6,Q24.8,14,20.000
7,M30.3,134,20.000
8,Q21.0,723,12.752
9,Q21.1,1158,7.586


## Preprocessing and reproducibility

The training loader draws a new start offset per ECG encounter, bounded by record length. A short recording is zero-padded. Each present lead is z-scored within its window, mapped to the same 12 anatomical slots as the baseline. Validation checkpoint selection always uses the first 5000 samples. All code maps and labels are rebuilt from train/validation metadata and checked against the saved baseline validation labels.

In [3]:
RECORDS=ROOT/'Child_ECG/Child_ecg'
def make_x(signal,fields):
 x=np.zeros((12,N_SAMPLES),dtype=np.float32)
 for j,lead in enumerate(fields['sig_name']):
  k=LI[lead.upper()];v=signal[:,j].astype(np.float32);sd=v.std()
  x[k,:len(v)]=(v-v.mean())/sd if sd>1e-8 else 0
 return x
class ECGDataset(Dataset):
 def __init__(self,rows,random_window=False):self.rows=rows;self.random_window=random_window
 def __len__(self):return len(self.rows)
 def __getitem__(self,i):
  r=self.rows[i];max_start=max(0,r['sampling_point']-N_SAMPLES)
  start=np.random.randint(0,max_start+1) if self.random_window else 0
  signal,fields=wfdb.rdsamp(str(RECORDS/r['filename']),sampfrom=start,sampto=min(start+N_SAMPLES,r['sampling_point']))
  assert fields['fs']==FS and np.isfinite(signal).all()
  return torch.from_numpy(make_x(signal,fields)),torch.from_numpy(r['target'])
class ThreeWindowDataset(Dataset):
 def __init__(self,rows):self.rows=rows
 def __len__(self):return len(self.rows)
 def __getitem__(self,i):
  r=self.rows[i];signal,fields=wfdb.rdsamp(str(RECORDS/r['filename']))
  assert fields['fs']==FS and np.isfinite(signal).all() and len(signal)==r['sampling_point']
  max_start=max(0,len(signal)-N_SAMPLES)
  starts=[0,max_start//2,max_start]
  x=np.stack([make_x(signal[z:z+N_SAMPLES],fields) for z in starts])
  return torch.from_numpy(x),torch.from_numpy(r['target'])
train_data=ECGDataset(train_rows,random_window=True);val_data=ECGDataset(val_rows)
val_loader=DataLoader(val_data,batch_size=BATCH_SIZE,shuffle=False,num_workers=2,pin_memory=True)
spec=importlib.util.spec_from_file_location('week5_resnet',ROOT/'NoteBooks/Week5/models.py')
module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module);ResNet1D=module.ResNet1D
common_bce=torch.nn.BCEWithLogitsLoss()
def seed_all():
 random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
 torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
def train_loader():
 g=torch.Generator().manual_seed(SEED)
 return DataLoader(train_data,batch_size=BATCH_SIZE,shuffle=True,num_workers=2,pin_memory=True,generator=g)
def validate(model):
 model.eval();tot=0.;ys=[];ps=[]
 with torch.no_grad():
  for x,y in val_loader:
   x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
   logits=model(x);tot+=common_bce(logits,y).item()*len(y)
   ys.append(y.cpu().numpy());ps.append(torch.sigmoid(logits).cpu().numpy())
 return tot/len(val_data),np.concatenate(ys),np.concatenate(ps)
config={'strategy':'uniform random 5000-sample training window; first-window checkpoint selection',
 'secondary_inference':'three deterministic windows at start, middle, end; mean or max probability',
 'architecture':'NoteBooks/Week5/models.py:ResNet1D(num_classes=16), random init',
 'seed':SEED,'split':'sorted patient IDs, seed42, floor 70/15/15','input_samples':N_SAMPLES,'lead_order':LEADS,
 'batch_size':BATCH_SIZE,'optimizer':'Adam','learning_rate':LR,'weight_decay':WEIGHT_DECAY,'max_epochs':MAX_EPOCHS,'patience':PATIENCE,
 'checkpoint_rule':'minimum first-window unweighted validation BCE; delta 1e-5','threshold':.5,
 'test_labels_or_waveforms_accessed':False,'torch':torch.__version__,'gpu':torch.cuda.get_device_name(0)}
(OUT/'config.json').write_text(json.dumps(config,indent=2)+'\n')
print('Random-window training rows',len(train_data),'validation rows',len(val_data))


Random-window training rows 9943 validation rows 2169


## Train one random-window model

Architecture, optimization, epoch cap, seed, and checkpoint rule match the baseline. This run changes training-window sampling only. The saved history includes training and validation loss for each epoch.

In [4]:
def run_candidate(name,criterion):
 seed_all();model=ResNet1D(num_classes=16).to(DEVICE)
 optimizer=torch.optim.Adam(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
 loader=train_loader();history=[];best=float('inf');best_epoch=None;stale=0
 path=OUT/f'{name}_best.pth'
 for epoch in range(1,MAX_EPOCHS+1):
  model.train();loss_sum=0.;bce_sum=0.
  for x,y in loader:
   x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
   logits=model(x);loss=criterion(logits,y)
   with torch.no_grad():bce_sum+=common_bce(logits,y).item()*len(y)
   optimizer.zero_grad(set_to_none=True);loss.backward();optimizer.step()
   loss_sum+=loss.item()*len(y)
  val_bce,vy,vp=validate(model)
  assert np.array_equal(vy,Y_val)
  history.append({'epoch':epoch,'training_objective':loss_sum/len(train_data),'training_unweighted_bce':bce_sum/len(train_data),'validation_unweighted_bce':val_bce})
  pd.DataFrame(history).to_csv(OUT/f'{name}_history.csv',index=False)
  if val_bce<best-1e-5:
   best=val_bce;best_epoch=epoch;stale=0
   torch.save({'model_state_dict':model.state_dict(),'architecture':'ResNet1D','num_classes':16,
               'base_codes_in_model_order':base_codes,'exact_to_base':base_of,'original_exact_codes':exact_codes,
               'strategy':name,'epoch':epoch,'validation_unweighted_bce':val_bce,'seed':SEED,
               'input_samples':N_SAMPLES,'lead_order':LEADS,'selection':'minimum first-window unweighted validation BCE'},path)
  else:stale+=1
  print(f'{name} epoch {epoch}: train objective {loss_sum/len(train_data):.5f}, val BCE {val_bce:.5f}, best {best_epoch}',flush=True)
  if stale>=PATIENCE:break
 saved=torch.load(path,map_location=DEVICE,weights_only=False)
 model.load_state_dict(saved['model_state_dict'])
 val_bce,vy,vp=validate(model)
 assert abs(val_bce-saved['validation_unweighted_bce'])<1e-6
 return {'epoch':best_epoch,'bce':val_bce,'y':vy,'p':vp,'checkpoint':path.name}

run=run_candidate('random_window',common_bce)
print('Random-window training complete',flush=True)


random_window epoch 1: train objective 0.10867, val BCE 0.06921, best 1


random_window epoch 2: train objective 0.06653, val BCE 0.06557, best 2


random_window epoch 3: train objective 0.06387, val BCE 0.06343, best 3


random_window epoch 4: train objective 0.06236, val BCE 0.06293, best 4


random_window epoch 5: train objective 0.06051, val BCE 0.06134, best 5


random_window epoch 6: train objective 0.05943, val BCE 0.06122, best 6


random_window epoch 7: train objective 0.05853, val BCE 0.06373, best 6


random_window epoch 8: train objective 0.05764, val BCE 0.06212, best 6


Random-window training complete


## Validation comparison

The first-window comparison includes the separately saved matched unweighted replication, which uses the same seed and loader setup as random-window training; it isolates the window-sampling change. The original first baseline is also retained as the project reference. Three-window inference is evaluated for both checkpoints with the same deterministic windows. Mean and max are separate exploratory aggregation rules; neither was tuned or selected. All metrics are calculated over the same 2169 validation records and 16 code targets. Per-target AP/AUROC are undefined when validation lacks positives.

In [5]:
baseline_p=prior[[f'probability::{x}' for x in base_codes]].to_numpy(dtype=float)
baseline_ckpt=torch.load(BASE/'resnet1d_base16_best.pth',map_location=DEVICE,weights_only=False)
assert baseline_ckpt['base_codes_in_model_order']==base_codes
baseline_model=ResNet1D(num_classes=16).to(DEVICE)
baseline_model.load_state_dict(baseline_ckpt['model_state_dict']);baseline_model.eval()
window_ckpt=torch.load(OUT/'random_window_best.pth',map_location=DEVICE,weights_only=False)
window_model=ResNet1D(num_classes=16).to(DEVICE)
window_model.load_state_dict(window_ckpt['model_state_dict']);window_model.eval()
three_loader=DataLoader(ThreeWindowDataset(val_rows),batch_size=16,shuffle=False,num_workers=2,pin_memory=True)
def three_probabilities(model):
 ys=[];ps=[];model.eval()
 with torch.no_grad():
  for x,y in three_loader:
   b,w,c,n=x.shape;x=x.reshape(b*w,c,n).to(DEVICE,non_blocking=True)
   p=torch.sigmoid(model(x)).reshape(b,w,16).cpu().numpy()
   ys.append(y.numpy());ps.append(p)
 return np.concatenate(ys),np.concatenate(ps)
by,b3=three_probabilities(baseline_model)
wy,w3=three_probabilities(window_model)
assert np.array_equal(by,Y_val) and np.array_equal(wy,Y_val)
assert np.allclose(b3[:,0,:],baseline_p,atol=1e-5)
assert np.allclose(w3[:,0,:],run['p'],atol=1e-5)
control=pd.read_csv(ROOT/'new work/Week7/experimental/imbalance_strategies/unweighted_replicate_validation_predictions.csv',dtype={'patient_id':str,'ecg_id':str})
assert control.patient_id.tolist()==prior.patient_id.tolist() and control.ecg_id.tolist()==prior.ecg_id.tolist()
control_p=control[[f'probability::{x}' for x in base_codes]].to_numpy(dtype=float)
probabilities={'baseline_first':baseline_p,'matched_unweighted_first':control_p,'random_window_first':run['p'],
 'baseline_three_mean':b3.mean(1),'random_window_three_mean':w3.mean(1),
 'baseline_three_max':b3.max(1),'random_window_three_max':w3.max(1)}
def metrics(y,p,threshold):
 pred=p>=threshold;flat_y=y.ravel().astype(int);flat_p=p.ravel();flat_pred=pred.ravel()
 tp=int(((flat_y==1)&flat_pred).sum());tn=int(((flat_y==0)&(~flat_pred)).sum())
 fp=int(((flat_y==0)&flat_pred).sum());fn=int(((flat_y==1)&(~flat_pred)).sum())
 aps=[];aucs=[];per=[]
 for j,c in enumerate(base_codes):
  yy=y[:,j].astype(int);pp=p[:,j];zz=pred[:,j]
  a=int(((yy==1)&zz).sum());b=int(((yy==0)&zz).sum());d=int(((yy==1)&(~zz)).sum());e=int(((yy==0)&(~zz)).sum())
  positive_patients=len({r['patient_id'] for r,positive in zip(val_rows,yy) if positive})
  ap=average_precision_score(yy,pp) if 0<yy.sum()<len(yy) else np.nan
  auc=roc_auc_score(yy,pp) if 0<yy.sum()<len(yy) else np.nan
  aps.append(ap);aucs.append(auc)
  per.append({'code':c,'positive_records':int(yy.sum()),'positive_patients':positive_patients,
              'tn':e,'fp':b,'fn':d,'tp':a,'sensitivity':a/(a+d) if a+d else np.nan,
              'specificity':e/(e+b) if e+b else np.nan,'precision':a/(a+b) if a+b else np.nan,
              'f1':2*a/(2*a+b+d) if a+d else np.nan,'auroc':auc,'average_precision':ap})
 return {'micro_tp':tp,'micro_fp':fp,'micro_fn':fn,'micro_tn':tn,'micro_sensitivity':tp/(tp+fn),
         'micro_specificity':tn/(tn+fp),'micro_precision':tp/(tp+fp) if tp+fp else np.nan,
         'micro_f1':2*tp/(2*tp+fp+fn),'micro_auroc':roc_auc_score(flat_y,flat_p),
         'micro_average_precision':average_precision_score(flat_y,flat_p),
         'macro_auroc':np.nanmean(aucs),'macro_average_precision':np.nanmean(aps),
         'macro_sensitivity':np.nanmean([z['sensitivity'] for z in per]),
         'macro_f1':np.nanmean([z['f1'] for z in per]),'subset_accuracy':float(np.mean(np.all(pred==y,axis=1))),
         'defined_ranking_targets':int(np.isfinite(aps).sum())},pd.DataFrame(per)
comparison=[];per_target=[]
for name,p in probabilities.items():
 m,table=metrics(Y_val,p,.5)
 comparison.append({'strategy':name,'threshold':.5,**m})
 table.insert(0,'strategy',name);per_target.append(table)
 d=prior[['patient_id','ecg_id']].copy()
 for j,c in enumerate(base_codes):d[f'probability::{c}']=p[:,j]
 d.to_csv(OUT/f'{name}_validation_predictions.csv',index=False)
comparison=pd.DataFrame(comparison);per_target=pd.concat(per_target,ignore_index=True)
comparison.to_csv(OUT/'validation_comparison.csv',index=False)
per_target.to_csv(OUT/'per_target_comparison.csv',index=False)
display(comparison.round(4))
print('All six validation strategies scored on identical records, 16 targets, and threshold 0.5')


,strategy,threshold,micro_tp,micro_fp,micro_fn,micro_tn,micro_sensitivity,micro_specificity,micro_precision,micro_f1,micro_auroc,micro_average_precision,macro_auroc,macro_average_precision,macro_sensitivity,macro_f1,subset_accuracy,defined_ranking_targets
0,baseline_first,0.5,147,67,524,33966,0.2191,0.9980,0.6869,0.3322,0.9131,0.3999,0.8283,0.1639,0.0752,0.1084,0.7621,14
1,matched_unweighted_first,0.5,125,49,546,33984,0.1863,0.9986,0.7184,0.2959,0.9133,0.3990,0.8375,0.1819,0.0460,0.0653,0.7616,14
2,random_window_first,0.5,125,38,546,33995,0.1863,0.9989,0.7669,0.2998,0.9141,0.3918,0.8334,0.1598,0.0491,0.0737,0.7649,14
3,baseline_three_mean,0.5,137,59,534,33974,0.2042,0.9983,0.6990,0.3160,0.9152,0.3951,0.8307,0.1632,0.0699,0.1039,0.7612,14
4,random_window_three_mean,0.5,117,35,554,33998,0.1744,0.9990,0.7697,0.2843,0.9163,0.3917,0.8335,0.1605,0.0397,0.0584,0.7626,14
5,baseline_three_max,0.5,159,81,512,33952,0.2370,0.9976,0.6625,0.3491,0.9150,0.3962,0.8339,0.1643,0.0794,0.1111,0.7630,14
6,random_window_three_max,0.5,136,52,535,33981,0.2027,0.9985,0.7234,0.3166,0.9168,0.3934,0.8439,0.1596,0.0530,0.0767,0.7658,14


All six validation strategies scored on identical records, 16 targets, and threshold 0.5


## Paired patient resampling

For the training-window comparison and the baseline three-window MAX comparison, 500 paired bootstrap resamples draw validation patients with replacement and retain all records from each sampled patient. The intervals describe sampling variability within this validation cohort. They do not remove the optimism from using this validation partition repeatedly during research.

In [6]:
# Paired patient bootstrap: exploratory uncertainty for two comparisons.
# Resample validation patients, keeping all their records together.
from collections import defaultdict
from sklearn.metrics import average_precision_score
patient_rows=defaultdict(list)
for i,r in enumerate(val_rows):patient_rows[r['patient_id']].append(i)
patients=np.array(list(patient_rows));rng=np.random.default_rng(20261104)
def small_metrics(y,p):
 z=p>=.5;tp=np.sum((y==1)&z);fp=np.sum((y==0)&z);fn=np.sum((y==1)&(~z))
 return np.array([2*tp/(2*tp+fp+fn),tp/(tp+fn),float(fp),average_precision_score(y.ravel(),p.ravel())])
pairs=[('random_window_first','matched_unweighted_first'),('baseline_three_max','baseline_first')]
rows=[]
for candidate,reference in pairs:
 observed=small_metrics(Y_val,probabilities[candidate])-small_metrics(Y_val,probabilities[reference])
 diffs=[]
 for _ in range(500):
  draw=rng.choice(patients,size=len(patients),replace=True)
  ix=np.concatenate([patient_rows[p] for p in draw]);yy=Y_val[ix]
  diffs.append(small_metrics(yy,probabilities[candidate][ix])-small_metrics(yy,probabilities[reference][ix]))
 diffs=np.asarray(diffs)
 for j,metric in enumerate(['micro_f1','micro_sensitivity','false_positives','micro_average_precision']):
  lo,hi=np.quantile(diffs[:,j],[.025,.975])
  rows.append({'candidate':candidate,'reference':reference,'metric':metric,'observed_delta':observed[j],
               'patient_bootstrap_95pct_low':lo,'patient_bootstrap_95pct_high':hi,'replicates':500})
uncertainty=pd.DataFrame(rows)
uncertainty.to_csv(OUT/'paired_patient_bootstrap.csv',index=False)
display(uncertainty.round(4))


,candidate,reference,metric,observed_delta,patient_bootstrap_95pct_low,patient_bootstrap_95pct_high,replicates
0,random_window_first,matched_unweighted_first,micro_f1,0.0039,-0.0224,0.0297,500
1,random_window_first,matched_unweighted_first,micro_sensitivity,0.0000,-0.0199,0.0185,500
2,random_window_first,matched_unweighted_first,false_positives,-11.0000,-24.0000,0.5250,500
3,random_window_first,matched_unweighted_first,micro_average_precision,-0.0072,-0.0272,0.0091,500
4,baseline_three_max,baseline_first,micro_f1,0.0169,0.0047,0.0301,500
5,baseline_three_max,baseline_first,micro_sensitivity,0.0179,0.0080,0.0282,500
6,baseline_three_max,baseline_first,false_positives,14.0000,7.0000,21.0000,500
7,baseline_three_max,baseline_first,micro_average_precision,-0.0037,-0.0101,0.0025,500


## Interpretation limits

Random-window augmentation changes which part of a record is seen during training, while labels apply to the full linked record. A window can omit features associated with an annotation. Repeated ECGs from the same patient are not independent. Mean/max scores may differ in calibration; fixed-threshold comparisons must be read together with AP, AUROC, precision, sensitivity, specificity, F1, and confusion counts. Any apparent gain is exploratory because this same validation partition guided the research. Rare and zero-positive targets cannot support robust per-code conclusions.

## Observed results and decision

The random-window checkpoint is epoch **6** (first-window validation BCE **0.06122**, versus **0.06112** for the untouched original). At threshold 0.5, the matched unweighted replication yields **TP 125 / FP 49 / FN 546**, F1 **0.296**, and micro AP **0.399**. Random-window training yields **TP 125 / FP 38 / FN 546**, F1 **0.300**, and micro AP **0.392**. Its F1 difference from the matched control is **+0.0039** (paired patient bootstrap 95% interval **−0.0224 to +0.0297**), so no reliable training improvement is established. Compared with the original baseline, it also loses sensitivity and AP.

On the **unchanged original checkpoint**, three-window MAX produces **TP 159 / FP 81 / FN 512**, F1 **0.349**, sensitivity **0.237**, micro AP **0.396**, and macro AP **0.164**. The first-window baseline is **TP 147 / FP 67 / FN 524**, F1 **0.332**, sensitivity **0.219**, micro AP **0.400**, and macro AP **0.164**. The MAX-versus-first paired patient bootstrap gives F1 difference **+0.0169** (95% interval **+0.0047 to +0.0301**) and **14** more false positives (interval **+7 to +21**). Three-window mean does not improve F1. These are validation-only observations after exploring two aggregation rules; they are not independent confirmation, and the AP difference is not a gain.

**Decision:** preserve the random-window checkpoint and history as exploratory results; retain the original Week 7 multilabel checkpoint as the primary model. Three-window MAX is a candidate operating/inference strategy for later prespecified evaluation, contingent on whether the added false positives are acceptable. Per-code metrics, including the many sparse targets, are in `per_target_comparison.csv`; patient-bootstrap results are in `paired_patient_bootstrap.csv`.